In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

**Load Project Utilities & Initialize Notebook Widgets**

In [0]:
%run /Workspace/Users/lhungen@gmail.com/FMCG_pipeline/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

bronze silver gold


In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://fmcg-databricks-bronze/{data_source}'
landing_path = f"{base_path}/landing"
processed_path = f"{base_path}/processed"
print(base_path)
print(landing_path)
print(processed_path)

#define the tables
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.{data_source}"



s3://fmcg-databricks-bronze/orders
s3://fmcg-databricks-bronze/orders/landing
s3://fmcg-databricks-bronze/orders/processed


## Bronze

In [0]:
df = spark.read.options(header=True, inferSchema=True).csv(f"{landing_path}/*.csv").withColumn("read_timestamp", F.current_timestamp()).select("*", "_metadata.file_name", "_metadata.file_size")

print("Total Rows: ", df.count())
df.show(5)

Total Rows:  51810
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FOCT62720602|Tuesday, Septembe...|     ABC987|  25891301|     71.0|2026-04-06 19:31:...|orders_2025_09_30...|    41446|
|FOCT62720602|Tuesday, Septembe...|     789720|  25891502|    125.0|2026-04-06 19:31:...|orders_2025_09_30...|    41446|
|FOCT62720602|Tuesday, Septembe...|     789720|  25891403|    462.0|2026-04-06 19:31:...|orders_2025_09_30...|    41446|
|FOCT62720602|Tuesday, Septembe...|    INVALID|  25891601|    133.0|2026-04-06 19:31:...|orders_2025_09_30...|    41446|
|FOCT62720602|Tuesday, Septembe...|     789720|  25891602|     79.0|2026-04-06 19:31:...|orders_2025_09_30...|    41446|
+------------

In [0]:
df.write\
    .format("delta") \
    .option('delta.enableChangeDataFeed','true')\
    .mode("append") \
    .saveAsTable(bronze_table)

   


### Moving files from source to processed directory

In [0]:
# This code moves files from the landing_path to the processed_path in the Databricks File System (DBFS).
# Since the paths are S3 URIs, it moves files within AWS S3 storage using Databricks utilities.
files = dbutils.fs.ls(landing_path)
for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{processed_path}/{file_info.name}",
        True
    )

## Silver

In [0]:
df_orders = spark.sql(f"SELECT * FROM {bronze_table}")
df_orders.show(2)

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FJUL33320501|          2025/07/01|     789320|  25891203|    150.0|2026-04-06 19:31:...|orders_2025_07_01...|    20744|
|FJUL33320501|          2025/07/01|     789320|  25891301|     46.0|2026-04-06 19:31:...|orders_2025_07_01...|    20744|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
only showing top 2 rows


#### Data Type Transaformation

In [0]:
# 1. Keep only rows where order_qty is present
df_orders = df_orders.filter(F.col("order_qty").isNotNull())


# 2. Clean customer_id → keep numeric, else set to 999999
df_orders = df_orders.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
     .otherwise("999999")
     .cast("string")
)

# 3. Remove weekday name from the date text
#    "Tuesday, July 01, 2025" → "July 01, 2025"
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

# 4. Parse order_placement_date using multiple possible formats
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
        F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
    )
)

# 5. Drop duplicates
df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

# 5. convert product id to string
df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))

In [0]:
# check what's the maximum and minimum date
df_orders.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2025-07-01|2025-11-30|
+----------+----------+



Join with Product

In [0]:
df_products = spark.table("fmcg.silver.products")
df_joined = df_orders.join(df_products, on="product_id", how="inner").select(df_orders["*"], df_products["product_code"])

df_joined.show(5)

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+--------------------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|        product_code|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+--------------------+
|FJUL32202603|          2025-07-01|     789202|  25891103|    370.0|2026-04-06 19:31:...|orders_2025_07_01...|    20744|102628255d24304d6...|
|FJUL33321602|          2025-07-01|     789321|  25891602|     74.0|2026-04-06 19:31:...|orders_2025_07_01...|    20744|778c2a7aa27bfdb21...|
|FJUL34303602|          2025-07-01|     789303|  25891102|    317.0|2026-04-06 19:31:...|orders_2025_07_01...|    20744|e92c739a8d78cd6cb...|
|FJUL34622602|          2025-07-01|     789622|  25891402|    439.0|2026-04-06 19:31:...|orders_2025_07_01...|    20744|fe5a8036be4b9a787...|
|FJUL3

# Gold

In [0]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM {silver_table};")

df_gold.show(2)

+------------+----------+-------------+--------------------+----------+-------------+
|    order_id|      date|customer_code|        product_code|product_id|sold_quantity|
+------------+----------+-------------+--------------------+----------+-------------+
|FJUL32202603|2025-07-01|       789202|102628255d24304d6...|  25891103|        370.0|
|FJUL33321602|2025-07-01|       789321|778c2a7aa27bfdb21...|  25891602|         74.0|
+------------+----------+-------------+--------------------+----------+-------------+
only showing top 2 rows


In [0]:
if not (spark.catalog.tableExists(gold_table)):
    print("creating New Table")
    df_gold.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias("source").merge(df_gold.alias("gold"), "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

creating New Table


# Merge with Parent Company
- Note: We want data for monthly level but child data is on daily level

In [0]:
df_child = spark.sql(f"SELECT date, product_code, customer_code, sold_quantity FROM {gold_table}")
df_child.show(10)

+----------+--------------------+-------------+-------------+
|      date|        product_code|customer_code|sold_quantity|
+----------+--------------------+-------------+-------------+
|2025-09-08|716fa4e54b7894c91...|       789903|         86.0|
|2025-09-08|062f5574bbdf4386b...|       789903|        121.0|
|2025-09-08|d9ebd1ca64d23951a...|       789903|         53.0|
|2025-09-08|451f7167b28a25bde...|       789902|         77.0|
|2025-09-08|778c2a7aa27bfdb21...|       789902|        124.0|
|2025-09-07|ee1f7df9cf660ef02...|       789902|        136.0|
|2025-09-08|77b6f538a9d0e0cf8...|       789902|        281.0|
|2025-09-06|716fa4e54b7894c91...|       999999|        139.0|
|2025-09-06|ee1f7df9cf660ef02...|       789720|        139.0|
|2025-09-07|716fa4e54b7894c91...|       999999|        138.0|
+----------+--------------------+-------------+-------------+
only showing top 10 rows


In [0]:
df_child.count()

40811

In [0]:
df_monthly = (
    df_child
    # 1. Get month start date (e.g., 2025-11-30 → 2025-11-01)
    .withColumn("month_start", F.trunc("date", "MM"))   # or F.date_trunc("month", "date").cast("date")

    # 2.Group at monthly grain by month_start + product_code + customer_code
    .groupBy("month_start", "product_code", "customer_code")
    .agg(
        F.sum("sold_quantity").alias("sold_quantity")
    )

    # 3. Rename month_start back to `date` to match your target schema
    .withColumnRenamed("month_start", "date")
)

df_monthly.show(5, truncate=False)

+----------+----------------------------------------------------------------+-------------+-------------+
|date      |product_code                                                    |customer_code|sold_quantity|
+----------+----------------------------------------------------------------+-------------+-------------+
|2025-09-01|716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742|789903       |1434.0       |
|2025-09-01|062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379|789903       |2555.0       |
|2025-09-01|d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5|789903       |855.0        |
|2025-09-01|451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa|789902       |2222.0       |
|2025-09-01|778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6|789902       |1254.0       |
+----------+----------------------------------------------------------------+-------------+-------------+
only showing top 5 rows


In [0]:
df_monthly.count()

3060

In [0]:
# This code merges monthly aggregated sales data (df_monthly) into the Delta table fact_orders.
# It updates existing rows if a match is found on date, product_code, and customer_code,
# and inserts new rows if no match exists.

gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders")
gold_parent_delta.alias("parent_gold").merge(
    df_monthly.alias("child_gold"),
    "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]